# 03 — Métaclasses

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre que `type` est la métaclasse par défaut ;
- écrire une métaclasse qui contrôle la création de classes ;
- distinguer `__new__` et `__init__` au niveau classe ;
- reconnaître les vrais cas d'usage (ABC, ORMs type SQLAlchemy) ;
- savoir **pourquoi** et **quand** préférer une alternative moins intrusive.


## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- le modèle objet complet (héritage, MRO, `super`, méthodes spéciales) ;
- les type hints modernes (`int | None`, génériques, `Protocol`, `TypeVar`) ;
- les dataclasses (`@dataclass`, `field`, `frozen=True`, `slots=True`) ;
- les décorateurs de fonction et de classe, et les gestionnaires de contexte ;
- les tests avec `pytest` (fixtures, paramétrage, monkeypatch) ;
- le packaging avec `pyproject.toml` et `uv` ;
- le logging (module `logging`, handlers, formatters) ;
- les bases de SQL et `sqlite3`, les expressions régulières.
- descripteurs et `__init_subclass__` (notebooks précédents).

Notions que nous allons **introduire ou approfondir** ici :

- `type` vu comme métaclasse et pas seulement comme fonction ;
- le protocole `__new__` / `__init__` au niveau metaclass ;
- les cas réels (SQLAlchemy, abc.ABCMeta) et leur justification.


## Plan

1. `type` : la fonction et la métaclasse
2. Créer une classe dynamiquement avec `type(name, bases, dict)`
3. Écrire sa propre métaclasse
4. `__new__` vs `__init__` au niveau métaclasse
5. `__prepare__` (ordered namespace)
6. Cas réel 1 — `abc.ABCMeta`
7. Cas réel 2 — l'approche ORM déclaratif (SQLAlchemy-like)
8. Pourquoi ne pas en abuser
9. Synthèse et arbre de décision
10. Exercices


---

## 1. `type` : fonction et métaclasse

Vous utilisez `type(obj)` depuis toujours pour connaître le type d'un objet. Mais `type` est aussi **la classe** de toutes les classes. Autrement dit : une classe est une **instance de `type`**.


In [ ]:
class Vide: pass

print(type(42))         # <class 'int'>
print(type("a"))        # <class 'str'>
print(type(Vide))        # <class 'type'>
print(type(Vide()))      # <class '__main__.Vide'>

Python est donc un langage où *les classes sont des objets de première classe* : on peut les passer en argument, les stocker, les créer dynamiquement à la volée.

---

## 2. Créer une classe dynamiquement avec `type(name, bases, dict)`

Dans sa forme à trois arguments, `type` est un **constructeur de classe**. Il prend :

- le **nom** de la classe (`str`),
- un **tuple de classes parentes** (bases),
- un **dictionnaire** des attributs et méthodes.

C'est exactement ce que fait Python quand vous écrivez `class Foo(Bar): ...` — la syntaxe `class` est du sucre pour un appel à `type(...)`.

In [ ]:
def hello(self):
    return f"hello from {self.nom}"

Utilisateur = type("Utilisateur", (object,), {
    "nom": "Ada",
    "hello": hello,
})

u = Utilisateur()
print(u.hello())
print(type(u).__name__)

La syntaxe `class` est équivalente à cet appel — juste plus lisible. Quand on écrit :

```python
class Point:
    def __init__(self, x): self.x = x
```

Python exécute le corps dans un namespace local, puis appelle `type("Point", (object,), ns)`.

---

## 3. Écrire sa propre métaclasse

Pour personnaliser la création de classe, on **hérite de `type`** et on précise `metaclass=MaMeta` sur la ligne `class`.


In [ ]:
class LogMeta(type):
    def __new__(mcs, name, bases, namespace):
        print(f"[meta] création de {name} avec {list(namespace)}")
        return super().__new__(mcs, name, bases, namespace)

class Service(metaclass=LogMeta):
    def ping(self):
        return "pong"

print(Service().ping())

Ce qui se passe :

1. Python lit le corps de `Service`, prépare un dict `namespace`.
2. Python trouve `metaclass=LogMeta` → appelle `LogMeta.__new__(LogMeta, "Service", (object,), namespace)`.
3. `super().__new__` (c.-à-d. `type.__new__`) construit l'objet classe.
4. La classe est renvoyée et assignée au nom `Service`.

La métaclasse est **héritée** : si `Service` a pour métaclasse `LogMeta`, alors toute sous-classe de `Service` aura aussi `LogMeta` pour métaclasse.

---

## 4. `__new__` vs `__init__` au niveau métaclasse

Comme toute classe, une métaclasse peut implémenter :

- `__new__(mcs, name, bases, namespace, **kwargs)` — appelée **avant** que l'objet classe n'existe.   Elle peut **transformer** le namespace avant la construction.
- `__init__(cls, name, bases, namespace, **kwargs)` — appelée **après** : `cls` est la classe déjà   créée. On peut encore la modifier mais l'objet est déjà vivant.

En pratique, la plupart du travail se fait dans `__new__` ou dans `__init__` selon le style. SQLAlchemy fait la majorité du travail dans `__init__`. L'important est de bien comprendre **à quel moment** chaque hook intervient.

In [ ]:
class TracePhases(type):
    def __new__(mcs, name, bases, ns):
        print(f"[new ] avant construction de {name}")
        return super().__new__(mcs, name, bases, ns)

    def __init__(cls, name, bases, ns):
        print(f"[init] après construction de {name}")
        super().__init__(name, bases, ns)

class X(metaclass=TracePhases):
    pass

### `__call__` — créer les instances

`MaMeta.__call__(cls, *args, **kwargs)` est appelée **quand on instancie la classe** (`MaClasse(...)`), pas quand on crée la classe. C'est ici qu'on peut implémenter un pattern singleton ou un cache.

In [ ]:
class SingletonMeta(type):
    _instances: dict = {}
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Config(metaclass=SingletonMeta):
    def __init__(self) -> None:
        self.debug = False

c1 = Config(); c1.debug = True
c2 = Config()
print(c1 is c2, c2.debug)

---

## 5. `__prepare__` — namespace personnalisé *(deep dive)*

Avant Python 3.7, les dictionnaires n'étaient pas ordonnés. Pour avoir un corps de classe ordonné (important pour un ORM qui veut connaître l'ordre de déclaration des champs), on utilisait `__prepare__` pour renvoyer un `OrderedDict`. Aujourd'hui c'est le comportement par défaut, mais `__prepare__` reste utile pour retourner **un dict spécialisé** (par exemple un dict qui refuse les doublons de nom).

In [ ]:
class NoDupDict(dict):
    def __setitem__(self, key, value):
        if key in self and not key.startswith("__"):
            raise TypeError(f"nom {key!r} défini plusieurs fois")
        super().__setitem__(key, value)

class NoDupMeta(type):
    @classmethod
    def __prepare__(mcs, name, bases, **kw):
        return NoDupDict()
    def __new__(mcs, name, bases, ns):
        return super().__new__(mcs, name, bases, dict(ns))

try:
    class Mauvais(metaclass=NoDupMeta):
        def f(self): return 1
        def f(self): return 2
except TypeError as e:
    print("Refusé :", e)


---

## 6. Cas réel — `abc.ABCMeta`

Le module `abc` utilise une métaclasse (`ABCMeta`) pour implémenter les classes abstraites :

- Elle **empêche l'instanciation** d'une classe qui contient des méthodes marquées   `@abstractmethod` non redéfinies.
- Elle permet l'**enregistrement virtuel** (`.register(SomeClass)`) — une classe peut être   considérée comme sous-classe sans héritage réel.
- `ABC` est simplement `class ABC(metaclass=ABCMeta): pass`.


In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Square(Shape):
    def __init__(self, c: float) -> None: self.c = c
    def area(self) -> float: return self.c * self.c

print(Square(3).area())

# Instancier Shape directement est refusé par la métaclasse
try:
    Shape()
except TypeError as e:
    print("Refusé :", e)

Techniquement, `ABCMeta.__call__` vérifie si la classe a encore des `__abstractmethods__`, et si oui, refuse l'instanciation. C'est un cas où **rien d'autre qu'une métaclasse ne fonctionne** (on doit intercepter l'instanciation d'une classe pas encore définie).

---

## 7. Cas réel — ORM déclaratif (SQLAlchemy-like)

SQLAlchemy, Django ORM, pydantic v1 (historiquement) utilisent des métaclasses pour construire automatiquement une table à partir d'une classe déclarative. Voici un mini-exemple.

In [ ]:
class Column:
    def __init__(self, type_: str, primary_key: bool = False) -> None:
        self.type_ = type_
        self.primary_key = primary_key
        self.name = None

    def __set_name__(self, owner, name: str) -> None:
        self.name = name

class ModelMeta(type):
    def __new__(mcs, name, bases, ns):
        cls = super().__new__(mcs, name, bases, ns)
        cls.__columns__ = [v for v in ns.values() if isinstance(v, Column)]
        cls.__table__ = name.lower()
        return cls

    def ddl(cls) -> str:
        cols = ", ".join(f"{c.name} {c.type_}" + (" PRIMARY KEY" if c.primary_key else "")
                        for c in cls.__columns__)
        return f"CREATE TABLE {cls.__table__} ({cols})"

class Base(metaclass=ModelMeta):
    pass

class User(Base):
    id = Column("INT", primary_key=True)
    name = Column("TEXT")
    email = Column("TEXT")

print(User.ddl())

**Note importante :** aujourd'hui, `User.ddl()` fonctionne parce que `ddl` est définie sur la **métaclasse**, pas sur `Base`. C'est le genre de subtilité qui fait que les métaclasses sont puissantes *et* délicates : les méthodes de la métaclasse deviennent des méthodes **de classe** (accessibles sur la classe, pas sur les instances).

Cet exemple est identique en esprit à SQLAlchemy 1.x. SQLAlchemy 2.x utilise aussi des métaclasses mais tire parti de `dataclass_transform` (PEP 681) pour que les IDE comprennent le typage.

---

## 8. Pourquoi (presque toujours) ne pas en abuser

Tim Peters : *« Metaclasses are deeper magic than 99 % of users should ever worry about. If you wonder whether you need them, you don't. »*

Raisons concrètes d'éviter :

- **Conflits d'héritage multiple.** Si deux classes parentes ont des métaclasses différentes,   Python refuse l'héritage avec `metaclass conflict`.
- **Lisibilité.** Le lecteur doit savoir ce qu'est une métaclasse, ce qu'elle fait, et quand.
- **Debuggabilité.** Les erreurs deviennent cryptiques.
- **Alternatives modernes.** `__init_subclass__`, décorateurs de classe, `@dataclass`,   `typing.dataclass_transform` couvrent presque tous les cas.

### Arbre de décision

```
Besoin d'intervenir à la création de classes ?
├─ Une seule classe ? → décorateur de classe
├─ Toutes les sous-classes d'une base ? → __init_subclass__
├─ Modifier l'instanciation (__call__) ? → encore __init_subclass__ + __init__
├─ Contrôler le namespace (refuser les doublons, etc.) ? → __prepare__ sur metaclass
└─ Intégrer avec un écosystème existant (ABCMeta, SQLAlchemy) ? → metaclass
```


---

## 9. Synthèse

| Point | À retenir |
|---|---|
| `type(obj)` vs `type(name, bases, dict)` | Deux usages de la même classe |
| Une classe est instance de sa métaclasse | et la métaclasse est (presque toujours) `type` |
| `__new__` au niveau meta | avant la construction : peut transformer le namespace |
| `__init__` au niveau meta | après : on peut enrichir |
| `__call__` au niveau meta | intercepte l'instanciation (singleton, cache) |
| `ABCMeta` et les ORMs | les **vrais** cas d'usage |
| `__init_subclass__` | l'alternative qui couvre 95 % des besoins |


---

## 10. Exercices

### Exercice 1 — Métaclasse qui force les noms en minuscules *(facile)*

Écrire une métaclasse qui **lève une erreur** si une classe qu'elle crée a des méthodes (noms callables non-`__dunder__`) qui ne sont pas en minuscules.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Metaclasses", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
class LowercaseMeta(type):
    def __new__(mcs, name, bases, ns):
        for k, v in ns.items():
            if callable(v) and not k.startswith("__") and k != k.lower():
                raise TypeError(f"nom {k!r} doit être en minuscules")
        return super().__new__(mcs, name, bases, ns)

class Good(metaclass=LowercaseMeta):
    def bonjour(self): return 42

try:
    class Bad(metaclass=LowercaseMeta):
        def Bonjour(self): return 42
except TypeError as e:
    print("refusé :", e)
```

</details>


### Exercice 2 — Métaclasse de comptage d'instances *(moyen)*

Écrire une métaclasse `CountMeta` qui ajoute automatiquement à chaque classe :

- un compteur de classe `_count: int`,
- un `__init__` wrappé qui incrémente le compteur à chaque instanciation,
- un classmethod `count()` qui le lit.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Metaclasses", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import functools

class CountMeta(type):
    def __new__(mcs, name, bases, ns):
        ns["_count"] = 0
        orig_init = ns.get("__init__", lambda self, *a, **k: None)

        @functools.wraps(orig_init)
        def __init__(self, *args, **kwargs):
            type(self)._count += 1
            orig_init(self, *args, **kwargs)

        ns["__init__"] = __init__
        ns["count"] = classmethod(lambda cls: cls._count)
        return super().__new__(mcs, name, bases, ns)

class Widget(metaclass=CountMeta):
    def __init__(self, nom: str) -> None:
        self.nom = nom

Widget("a"); Widget("b"); Widget("c")
print(Widget.count())
```

</details>


### Exercice 3 — Réécrire le mini-ORM **sans** métaclasse *(difficile)*

Reprendre l'exemple `ModelMeta` de la section 7 et le réécrire en utilisant `__init_subclass__` au lieu d'une métaclasse. Identifier ce qu'on perd (s'il y a quelque chose).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Metaclasses", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
class Column:
    def __init__(self, type_: str, primary_key: bool = False) -> None:
        self.type_, self.primary_key, self.name = type_, primary_key, None
    def __set_name__(self, owner, name): self.name = name

class Base:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.__columns__ = [v for v in vars(cls).values() if isinstance(v, Column)]
        cls.__table__ = cls.__name__.lower()

    @classmethod
    def ddl(cls) -> str:
        cols = ", ".join(f"{c.name} {c.type_}" + (" PRIMARY KEY" if c.primary_key else "")
                        for c in cls.__columns__)
        return f"CREATE TABLE {cls.__table__} ({cols})"

class User(Base):
    id = Column("INT", primary_key=True)
    name = Column("TEXT")

print(User.ddl())
# Ce qu'on perd : rien d'utile dans ce cas. Le seul cas où __init_subclass__
# ne suffit pas, c'est quand on doit modifier __call__ ou __prepare__.
```

</details>


### Exercice 4 — Détection de conflit de métaclasses *(deep dive)*

Créer deux métaclasses `MetaA` et `MetaB` différentes, créer deux classes de base utilisant chacune une métaclasse, puis tenter d'hériter des deux. Observer l'erreur `metaclass conflict` et expliquer comment la résoudre (créer une troisième métaclasse qui hérite des deux).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Metaclasses", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
class MetaA(type): pass
class MetaB(type): pass

class A(metaclass=MetaA): pass
class B(metaclass=MetaB): pass

try:
    class C(A, B): pass
except TypeError as e:
    print("conflit :", e)

# Solution : une métaclasse fille commune
class MetaAB(MetaA, MetaB): pass
class C(A, B, metaclass=MetaAB): pass
print("OK :", C)
```

</details>


---

## Ressources

- [docs Python — data model (metaclasses)](https://docs.python.org/3/reference/datamodel.html#metaclasses)
- [PEP 3115 — metaclass `__prepare__`](https://peps.python.org/pep-3115/)
- [PEP 681 — `dataclass_transform`](https://peps.python.org/pep-0681/)
- *Fluent Python* — chapitre 24, *Class metaprogramming*
- SQLAlchemy source : `lib/sqlalchemy/orm/decl_base.py`
